In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from omegaconf import OmegaConf
from typing import Tuple, Dict, Optional

ROOT_DIR = Path("..").resolve()
sys.path.append(str(ROOT_DIR))

from src.utils.processing_utils import clip
from src.utils.debug import print_images_statistics

# --- CONFIGURATION ---
RESULTS_DIR = ROOT_DIR / "results/fpga/active_model/results/"

# Normal lambda = 10
# RESULTS_DIR = ROOT_DIR / "results/fpga/FPGA_inference_ResSHyp-relu_s2_L1000_pt_2026-02-18_02-15-54"
# lambda=10 with --fast-finetune
# RESULTS_DIR = ROOT_DIR / "results/fpga/inference_ResidualScaleHyperpriorDPUWrapper_pt_2021-11-21_17-02-34"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory '{RESULTS_DIR}' does not exist.")


In [ ]:
# 1. Print Execution Summary
metrics_path = RESULTS_DIR / "metrics.json"
print(metrics_path.resolve())
log_path = RESULTS_DIR / "inference.log"

# Check Log Header for Model Info
if log_path.exists():
    print("\n--- Run Info ---")
    with open(log_path, "r") as f:
        for line in f:
            if "Model" in line or "DPU" in line or "Inference" in line:
                print(line.strip())

# Print Metrics
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print("\n--- Final Metrics ---")
    print(json.dumps(metrics, indent=4))
else:
    print("\nmetrics.json not found in result directory.")

In [ ]:
recons_dir = RESULTS_DIR / "reconstructions_test_set"
vis_noisy = np.load(recons_dir / "vis_noisy.npy")
vis_recon = np.load(recons_dir / "vis_recon.npy")
vis_adam = np.load(recons_dir / "vis_adam.npy")
vis_merlin = np.load(recons_dir / "vis_merlin.npy")

print(f"Loaded {len(vis_noisy)} patches for visualization.")

# --- Visualization Logic (Inline to ensure correctness) ---
num_patches = len(vis_noisy)
N = min(5, num_patches)

fig, axes = plt.subplots(4, N, figsize=(4 * N, 16))
if N == 1:
    axes = axes.reshape(4, 1)

for i in range(N):
    # Clip individual patches (not the whole batch)
    noisy_disp = clip(vis_noisy[i])
    recon_disp = clip(vis_recon[i])
    adam_disp = clip(vis_adam[i])
    merlin_disp = clip(vis_merlin[i])

    # Row 0: Original Noisy (LogI)
    axes[0, i].imshow(noisy_disp, cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Noisy Input")

    # Row 1: Reconstruction (LogI)
    axes[1, i].imshow(recon_disp, cmap="gray")
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("Reconstruction")

    # Row 2: ADAM NOC GT (LogI)
    axes[2, i].imshow(adam_disp, cmap="gray")
    axes[2, i].axis("off")
    if i == 0:
        axes[2, i].set_title("ADAM NOC GT")

    # Row 3: MERLIN GT (LogI)
    axes[3, i].imshow(merlin_disp, cmap="gray")
    axes[3, i].axis("off")
    if i == 0:
        axes[3, i].set_title("MERLIN GT")

plt.tight_layout()
plt.show()
fig.savefig(RESULTS_DIR / "reconstructions_visualization.png")

In [ ]:
def obtain_tile_reconstruction_and_metrics_from_training(results_dir: Path) -> Tuple[Optional[np.ndarray], Dict[str, float]]:
    """"""
    manifest_path = results_dir.parent / "manifest.json"
    # load the manifest and parse "original_run_dir"
    if manifest_path.exists():
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        original_run_dir_str = manifest.get("original_run_dir", "")
        if not original_run_dir_str:
            print("[!] 'original_run_dir' not found in manifest.json")
            return None, {}
        train_run_dir = Path(original_run_dir_str)
        
    # tmp fix if the original_run_dir Path was written in the Docker container we need to replace "worskpace" by "home/leon_ce/dev/Vitis-AI"
    if train_run_dir and "workspace" in str(train_run_dir):
        train_run_dir = Path(str(train_run_dir).replace("workspace", "home/leon_ce/dev/Vitis-AI"))
        print(f"Adjusted train_run_dir to: {train_run_dir}")

    gpu_metrics = {}
    if train_run_dir and train_run_dir.exists():
        # Look for 'reconstruction_image_*.png' in the test media folder
        test_images_dir = train_run_dir / "wandb/latest-run/files/media/images/test"
        if test_images_dir.exists():
            found_pngs = list(test_images_dir.glob("reconstruction_image_*.png"))
            if found_pngs:
                gpu_recon_path = found_pngs[0]
                # Load as grayscale numpy array
                gpu_recon_img = np.array(Image.open(gpu_recon_path).convert("L"))
                print(f"Loaded GPU Ref from: {gpu_recon_path.name}")
            else:
                print("[!] No reconstruction_image_*.png found in wandb/media/images/test")
                return None, {}

            # Find corresponding summary metrics for this reconstructions
            summary_path = train_run_dir / "wandb/latest-run/files/wandb-summary.json"
            if summary_path.exists():
                with open(summary_path, "r") as f:
                    summary_metrics = json.load(f)
                print("\n--- GPU Reconstruction Metrics from WandB Summary ---")
                large_patch_summary = summary_metrics["test/reconstruction_image"]
                gpu_ref_caption = large_patch_summary["caption"]
                # Parse 'bpp=' and 'psnr=' from caption
                gpu_metrics["bpp"] = float(gpu_ref_caption.split("bpp=")[1].split(",")[0])
                gpu_metrics["psnr_to_merlin_dds"] = float(gpu_ref_caption.split("psnr=")[1].split(",")[0])
                print(f"Parsed GPU Ref BPP: {gpu_metrics['bpp']:.4f}, PSNR: {gpu_metrics['psnr_to_merlin_dds']:.4f}dB")
        else:
            print(f"[!] Test images dir not found: {test_images_dir}")
            return None, {}
    else:
        print(f"[!] Original Run Dir does not exist on this machine: {train_run_dir}")
        return None, {}
        
    return gpu_recon_img, gpu_metrics

In [ ]:
# -----------------------------------------------------------------------------
# 4. Large Tile Visualization & Analysis
# -----------------------------------------------------------------------------

# Find all large_tiles that have been reconstructed. Look for *.npy files in RESULTS_DIR that match the pattern "*_recon_linA.npy" and extract the tile name.
all_recon_tiles = list(RESULTS_DIR.glob("*_recon_linA.npy"))
if not all_recon_tiles:
    raise FileNotFoundError(f"No reconstruction files found in {RESULTS_DIR} matching '*_recon_linA.npy'.")

for recon_tile_path in all_recon_tiles:
    # get tile name by removing suffix "_recon_linA.npy"
    tile_name = recon_tile_path.name.replace("_recon_linA.npy", "")
    print(f"Found reconstruction file for tile: {tile_name}")
    
    # Get the corresponding images and GTs from data/visualization/tile_name
    tile_data_dir = ROOT_DIR / "data" / "visualization" / tile_name
    if not tile_data_dir.exists():
        raise FileNotFoundError(f"Data directory for tile '{tile_name}' not found at {tile_data_dir}. Skipping this tile.")
    
    # Load corresponding metrics
    tile_metrics_path = RESULTS_DIR / f"{tile_name}_metrics.json"
    tile_stats = {}
    if tile_metrics_path.exists():
        with open(tile_metrics_path, "r") as f:
            tile_stats = json.load(f)
        print(f"Loaded Metrics from: {tile_metrics_path.name}")
    else:
        raise FileNotFoundError(f"Metrics file not found at: {tile_metrics_path}")

    # Load the reconstruction for this tile
    recon_linA = np.load(recon_tile_path)
    # Load the corresponding noisy input, ADAM GT, and MERLIN GT from the tile data directory
    noisy_linA = np.load(tile_data_dir / "linA_Noisy.npy")
    merlin_linA = np.load(tile_data_dir / "linA_MERLIN.npy")
    adam_noc_linA = np.load(tile_data_dir / "linA_ADAM_NOC.npy")
    merlin_dds_linA = np.load(tile_data_dir / "linA_MERLIN_DDS.npy")
    # Reconstruction (GPU) from WandB directory or Run Logic
    gpu_recon_logI, gpu_recon_metrics = obtain_tile_reconstruction_and_metrics_from_training(RESULTS_DIR)
    if not gpu_recon_metrics:
        print("[!] No GPU reconstruction metrics found. GPU image (if found) will be shown without metrics.")
        gpu_subtitle = f"\nBPP={gpu_recon_metrics['bpp']:.4f}|PSNR={gpu_recon_metrics['psnr_to_merlin_dds']:.4f}dB" if gpu_recon_metrics else ""
    

    # Print Statistics using new function
    images_map_stats = {
        "Noisy": noisy_linA,
        "Reconstruction": recon_linA,
        "ADAM_NOC": adam_noc_linA,
        "MERLIN": merlin_linA,
        "MERLIN_DDS": merlin_dds_linA,
        "GPU Recon (Ref)": gpu_recon_logI,
    }
    print_images_statistics(
        images_map_stats,
        metrics=["min", "max", "mean", "std"],
        title="Linear Amplitude Statistics (GPU is LogI 0-255)",
    )

# D. Plotting
# Layout: 2 Rows. Row 1: Inputs/GTs. Row 2: Reconstructions.
fig = plt.figure(figsize=(15, 12))
axs = [
    fig.add_subplot(2, 3, 1),  # Noisy
    fig.add_subplot(2, 3, 2),  # ADAM_NOC
    fig.add_subplot(2, 3, 3),  # MERLIN
    fig.add_subplot(2, 3, 4),  # MERLIN_DDS
    fig.add_subplot(2, 3, 5),  # FPGA Recon
    fig.add_subplot(2, 3, 6),  # GPU Recon
]

# Titles
bpp_val = tile_stats.get("bpp", float("nan"))
psnr_noisy = tile_stats.get("psnr_noisy", float("nan"))
psnr_adam = tile_stats.get("psnr_adam", float("nan"))
psnr_merlin = tile_stats.get("psnr_merlin", float("nan"))

plot_list = [
    (noisy_linA, "Noisy Input", True),
    (adam_noc_linA, f"ADAM_NOC GT\nPSNR: {psnr_adam:.4f}dB", True),
    (merlin_linA, "MERLIN GT", True),
    (merlin_dds_linA, "MERLIN_DDS GT", True),
    (
        recon_linA,
        f"Reconstruction (FPGA ZCU102)\nBPP={bpp_val:.4f}|PSNR={psnr_merlin:.4f}dB",
        True,
    ),
    (
        gpu_recon_logI,
        f"Reconstruction (GPU: NVIDIA RTX A4000){gpu_subtitle}",
        False,
    ),  # False = Don't process
]

for ax, (img, title, process) in zip(axs, plot_list):
    if img is not None:
        if process:
            # Convert to Log Intensity for Visualization
            # Log(Amplitude^2) = 2 * Log(Amplitude)
            img_vis = 2 * np.log(img + 1e-9)
            # Apply Clipping
            img_vis = clip(img_vis)
        else:
            # Already processed 2D array (GPU Image)
            img_vis = img

        # Determine colormap: Use gray if we processed it OR if it's a 2D array
        use_cmap = "gray" if (process or getattr(img_vis, "ndim", 0) == 2) else None

        im = ax.imshow(img_vis, cmap=use_cmap)
        ax.set_title(title, fontsize=11, fontweight="bold")

        # Only show colorbar for processed data OR 2D arrays (like our GPU gray image)
        if process or (getattr(img_vis, "ndim", 0) == 2):
            fig.colorbar(im, ax=ax, shrink=0.6)
    else:
        ax.text(0.5, 0.5, "Image Not Found", ha="center", va="center")

    ax.axis("off")

plt.tight_layout()
plt.show()